# LeNet-1 (1989) Implementation

This notebook is reimplementation of the original Convnet paper from LeCun et.al used for digit recognition in handwritten zip codes [^1]. The neural net was then evolved from this first version over a decade [^2].

## References
[^1]: LeCun, Y. et al. (1989). Backpropagation applied to Handwritten Zip Code Recognition. [Link to PDF](http://yann.lecun.com/exdb/publis/pdf/lecun-89e.pdf)

[^2]: LeCun, Y. et al. (1998) GradientBased Learning Applied to Document Recognition



In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)

## Table of Contents
1. [Data Preparation](#data-preparation)
2. [Model Architecture](#model-architecture)
3. [Forward Pass](#forward-pass)
4. [Training Loop](#training-loop)
5. [Results](#results)

## Data Preparation

The data pre-processing is done in the notebook **pytorch_playground**, which load, resizes to 16x16 image from 28x28 image in MNIST Dataset, and stores the subset of the dataset to npz file in data directory.
* The saved dataset contains 7291 training and 2007 test images of handwritten digits and corresponding labels (0-9)

In [2]:
data_train = np.load("./data/train1989.npz")
data_test = np.load("./data/test1989.npz")

# Access arrays inside
Xtr = torch.from_numpy(data_train["X"])
Ytr = torch.from_numpy(data_train["Y"])

Xte = torch.from_numpy(data_test["X"])
Yte = torch.from_numpy(data_test["Y"])

FileNotFoundError: [Errno 2] No such file or directory: './data/train1989.npz'

**Task** Check the shape of the training and test data and labels.

In [3]:
##TODO: Write code for checking shape.
import torch
from torchvision import datasets, transforms

transform = transforms.ToTensor()

# Step 2: Load MNIST dataset directly
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Step 3: Extract images and labels
train_images = torch.stack([img for img, lbl in train_dataset])
train_labels = torch.tensor([lbl for img, lbl in train_dataset])

test_images  = torch.stack([img for img, lbl in test_dataset])
test_labels  = torch.tensor([lbl for img, lbl in test_dataset])

# Step 4: Check shapes
print("Train images shape:", train_images.shape)
print("Train labels shape:", train_labels.shape)
print("Test images shape:", test_images.shape)
print("Test labels shape:", test_labels.shape)

Train images shape: torch.Size([60000, 1, 28, 28])
Train labels shape: torch.Size([60000])
Test images shape: torch.Size([10000, 1, 28, 28])
Test labels shape: torch.Size([10000])


## Model Architecture

Score function is the mathematical mapping that takes the input (e.g. image of handwritten digit) and transforms into scores for each possible class.  In other words, forward pass of a neural network is the score function. The simplest being $$f(x, W, b) = f_a(\sum_{i=1}^{n} w_i x_i + b)$$
* $W$: (weights) are the parameters of the neural network
* $x$: features chosen for the problem (in this case, grayscale value of pixels)
* $f_a$: chosen activation function for introducing non-linearity (e.g. tanh or relu)

### Score function: Weights and bias
The notebook uses paper implementation of the weight initialization which requires fan_in(number of input connections to a neuron) which prevents vanishing gradients.

In [15]:
def init_weights(fan_in, *shape):
    """Weight initialization as described in the paper"""
    weight = (torch.rand(*shape) - 0.5) * 2 * 2.4 / fan_in
    return nn.Parameter(weight)

**Task** you'll need to implement TODO section, with following specification for parameters
* H1 layer is a 2d Convolution layer with 12 filter with mask 5x5 and stride of 2
* H3 layer is fully connected layer (FC) where input shape is (1, 192) output shape is (1, 30)
* Final layer is also FC mapping to 10 digit classes
* activation function is tanh
* mac, act are the number of multiply-accumulate and activations, respectively.

In [6]:
import torch
import torch.nn as nn

def init_weights(fan_in, *shape):
    weight = (torch.rand(*shape) - 0.5) * 2 * 2.4 / fan_in
    return nn.Parameter(weight)

macs = 0
acts = 0

# H1
H1w = init_weights(5*5*1, 8, 1, 5, 5)
H1b = nn.Parameter(torch.zeros(8, 12, 12))

macs += 28800
acts += 1152

# H2
H2w = init_weights(5*5*8, 12, 8, 5, 5)
H2b = nn.Parameter(torch.zeros(12, 4, 4))

macs += 29760
acts += 192

# H3
H3w = init_weights(192, 30, 192)
H3b = nn.Parameter(torch.zeros(30))

macs += 192 * 30 + 30
acts += 30

# Output
outw = init_weights(30, 10, 30)
outb = nn.Parameter(-torch.ones(10))

macs += 30 * 10 + 10
acts += 10

assert macs == 64660

print("MACs:", macs)
print("Activations:", acts)

MACs: 64660
Activations: 1384


**Task** Find out the following features of the model architecture
   * Parameters:
   * macs:
   * activations:

Number of parameters: 10044  
Number of MACs: 64660  
Number of activations: 1384  Enter answer here

### Score function: Forward pass

The weights, bias, activation function in each layer of neural network define the forward pass. The forward pass is the inference and a major step in training.

**Task** you'll need to implement TODO section in forward pass in following cell, with following specification as in the paper
* H1 layer is a 2d Convolution layer with 12 filter with mask 5x5 and stride of 2. Padding is given.
* H3 layer is fully connected layer (FC) where input shape is (1, 192) output shape is (1, 30)
* Final layer is also FC mapping to 10 digit classes
* activation function is tanh

In [8]:
def forward(x):
    x = F.pad(x, (2, 2, 2, 2), mode='constant', value=-1.0)

    x = F.conv2d(x, H1w, stride=2) + H1b
    x = torch.tanh(x)

    x = F.pad(x, (2, 2, 2, 2), mode='constant', value=-1.0)
    slice1 = F.conv2d(x[:, 0:8], H2w[0:4], stride=2)
    slice2 = F.conv2d(x[:, 4:12], H2w[4:8], stride=2)
    slice3 = F.conv2d(torch.cat([x[:, 0:4], x[:, 8:12]], dim=1), H2w[8:12], stride=2)
    x = torch.cat((slice1, slice2, slice3), dim=1) + H2b
    x = torch.tanh(x)

    x = x.flatten(start_dim=1)

    x = F.linear(x, H3w, H3b)
    x = torch.tanh(x)

    x = F.linear(x, outw, outb)

    assert x.shape == (1, 10)
    return x

### Loss Function: Error

Loss function measures how good the model is in predicting. High loss means the score for wrong digit is high and right digit is low. It is used in ML training for optimizing the network parameters. Error is a more intuitive metric for judgment of overall training performance.

**Task** Update the loss function in following cell

In [9]:
def train_step(optimizer, x, y):
    optimizer.zero_grad()

    yhat = forward(x)
    y_onehot = F.one_hot(y, num_classes=10).float()
    loss = F.mse_loss(yhat, y_onehot)

    loss.backward()
    optimizer.step()

    return loss.item()

Following functions visualize and report the convergence of ML training

In [10]:
def eval_split(split, X_tr, Y_tr, X_te, Y_te):
    X, Y = (X_tr, Y_tr) if split == 'train' else (X_te, Y_te)
    yhat = forward(X)

    loss = torch.mean((Y - yhat)**2)
    err = torch.mean((Y.argmax(dim=1) != yhat.argmax(dim=1)).float())
    misses = int(err.item() * Y.shape[0])

    print(f"eval: split {split:5s}. loss {loss.item():e}. error {err.item()*100:.2f}%. misses: {misses}")
    return loss.item(), err.item(), misses

def plot_metrics(train_errs, test_errs):
    plt.figure(figsize=(10, 5))
    plt.plot(train_errs, label='Train Error', color='blue', marker='o')
    plt.plot(test_errs, label='Test Error', color='orange', marker='s')
    plt.title('Training vs Test Error')
    plt.xlabel('Epoch')
    plt.ylabel('Error (%)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

### Optimization: Model training

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def init_weights(fan_in, *shape):
    weight = (torch.rand(*shape) - 0.5) * 2 * 2.4 / fan_in
    return nn.Parameter(weight)

H1w = init_weights(5 * 5 * 1, 12, 1, 5, 5)
H1b = nn.Parameter(torch.zeros(12, 14, 14))

H2w = init_weights(5 * 5 * 8, 12, 8, 5, 5)
H2b = nn.Parameter(torch.zeros(12, 4, 4))

H3w = init_weights(192, 30, 192)
H3b = nn.Parameter(torch.zeros(30))

outw = init_weights(30, 10, 30)
outb = nn.Parameter(-torch.ones(10))

def forward(x):
    x = F.pad(x, (2, 2, 2, 2), mode='constant', value=-1.0)
    x = F.conv2d(x, H1w, stride=2) + H1b
    x = torch.tanh(x)
    x = F.avg_pool2d(x, kernel_size=2, stride=2)

    x = F.pad(x, (2, 2, 2, 2), mode='constant', value=-1.0)
    slice1 = F.conv2d(x[:, 0:8], H2w[0:4], stride=2)
    slice2 = F.conv2d(x[:, 4:12], H2w[4:8], stride=2)
    slice3 = F.conv2d(torch.cat([x[:, 0:4], x[:, 8:12]], dim=1), H2w[8:12], stride=2)
    x = torch.cat((slice1, slice2, slice3), dim=1) + H2b
    x = torch.tanh(x)

    x = x.flatten(start_dim=1)
    x = F.linear(x, H3w, H3b)
    x = torch.tanh(x)
    x = F.linear(x, outw, outb)
    return x

def train_step(optimizer, x, y):
    optimizer.zero_grad()
    yhat = forward(x)
    loss = F.mse_loss(yhat, y)
    loss.backward()
    optimizer.step()
    return loss.item()

def eval_split(split, Xtr, Ytr, Xte, Yte):
    if split == 'train':
        X, Y = Xtr, Ytr
    else:
        X, Y = Xte, Yte

    with torch.no_grad():
        losses = []
        misses = 0

        for i in range(X.shape[0]):
            x = X[i:i+1]
            y = Y[i:i+1]
            yhat = forward(x)
            losses.append(F.mse_loss(yhat, y).item())
            pred = yhat.argmax(dim=1)
            true = y.argmax(dim=1)
            misses += (pred != true).sum().item()

        loss = sum(losses) / len(losses)
        err = misses / X.shape[0]

    return loss, err, misses

Xtr = train_images
Xte = test_images
Ytr = F.one_hot(train_labels, num_classes=10).float()
Yte = F.one_hot(test_labels, num_classes=10).float()

num_epochs = 23
params = [H1w, H1b, H2w, H2b, H3w, H3b, outw, outb]
optimizer = torch.optim.SGD(params, lr=0.03)

history = {'train_err': [], 'test_err': []}

for pass_num in range(num_epochs):
    for step_num in range(Xtr.shape[0]):
        x = Xtr[step_num:step_num+1]
        y = Ytr[step_num:step_num+1]
        loss = train_step(optimizer, x, y)

    print(f"\nEpoch {pass_num + 1}/{num_epochs}")
    train_loss, train_err, train_misses = eval_split('train', Xtr, Ytr, Xte, Yte)
    test_loss, test_err, test_misses = eval_split('test', Xtr, Ytr, Xte, Yte)

    history['train_err'].append(train_err * 100)
    history['test_err'].append(test_err * 100)

    print(f"train loss: {train_loss:.4f}, train error: {train_err * 100:.2f}%")
    print(f"test loss: {test_loss:.4f}, test error: {test_err * 100:.2f}%")


Epoch 1/23
train loss: 0.0224, train error: 9.93%
test loss: 0.0216, test error: 9.17%

Epoch 2/23
train loss: 0.0120, train error: 5.44%
test loss: 0.0115, test error: 5.28%

Epoch 3/23
train loss: 0.0088, train error: 4.13%
test loss: 0.0083, test error: 3.95%

Epoch 4/23
train loss: 0.0071, train error: 3.40%
test loss: 0.0067, test error: 3.15%

Epoch 5/23
train loss: 0.0061, train error: 2.92%
test loss: 0.0058, test error: 2.64%

Epoch 6/23
train loss: 0.0055, train error: 2.66%
test loss: 0.0052, test error: 2.35%

Epoch 7/23
train loss: 0.0051, train error: 2.46%
test loss: 0.0048, test error: 2.22%

Epoch 8/23
train loss: 0.0047, train error: 2.31%
test loss: 0.0045, test error: 2.18%

Epoch 9/23
train loss: 0.0045, train error: 2.17%
test loss: 0.0043, test error: 2.14%

Epoch 10/23
train loss: 0.0042, train error: 2.05%
test loss: 0.0041, test error: 2.10%

Epoch 11/23
train loss: 0.0041, train error: 1.92%
test loss: 0.0040, test error: 2.04%

Epoch 12/23
train loss: 0.003

**Task** In 1989 the training of the CNN took few days.

* What is the training time of your implementation (number of epochs=23)
* Which layer of neural network needs most mac operations (arithmetic)
* How many bytes are transferred to/from memory by each layer considering the input output, and weights

Enter your answer here.

# Notes: Ideas on experiments
* Even though the 1989 LeCun paper used MSE, the modern standard for digit recognition is Cross-Entropy.
* different learning rates
* different optimizer: Adam
* Visualize the feature maps
* weight decay, regularization
* LeNet-1 Network Class in JAX
    * image batch: (7291, 20, 20, 1)→ interpreted as NHWC (batch, height, width, channels) in JAX. (7291, 1, 20, 20) in Pytorch.
    * weights: (12, 1, 5, 5)→ interpreted as (out_channels, in_channels, filter_height, filter_width)
    * In pytorch, NCHW is standard dimension format. you must tell JAX which dimension format you’re using. Correct usage: NHWC input, OIHW kernel. Output dimension same as bias